In [ ]:
"""
For Google Colab.
"""

# Clone repo
%cd /content
!git clone https://github.com/AHHHHHH0-0/Extending-CoFinDiff.git
%cd Extending-CoFinDiff

# Open terminal and switch branch

## Import


In [ ]:
import sys
import json
import torch
import tqdm
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

sys.path.insert(0, str(Path().resolve().parent.parent))
from denoiser.unet_model_ca import UNetDenoiserCA
from diffusion.diffusion_ca import DiffusionCA
from preprocessing.condition_encoder import ConditionEncoder
from preprocessing.haar_wavelet import HaarWaveletTransform
from config import project_config, diffusion_config, preprocess_config

## Config


In [ ]:
# Seed and device
SEED = project_config.SEED
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

DEVICE = project_config.DEVICE
# Paths relative to repo root (parent of notebooks/ca/)
REPO_ROOT = Path().resolve().parent.parent
CHECKPOINT_PATH = REPO_ROOT / "models" / "ca" / "checkpoints" / "best_model.pt"

print(f"Device: {DEVICE}")
print(f"Checkpoint: {CHECKPOINT_PATH}")

## Load Models


In [ ]:
# Models
denoiser = UNetDenoiserCA(in_channels=1).to(DEVICE)
diffusion = DiffusionCA()
haar = HaarWaveletTransform()

# Load checkpoint — build ConditionEncoder with stored normalization stats
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
cond_means = checkpoint['cond_norm_means']
cond_stds = checkpoint['cond_norm_stds']
cond_encoder = ConditionEncoder(cond_means=cond_means, cond_stds=cond_stds).to(DEVICE)

denoiser.load_state_dict(checkpoint['model_state_dict'])
encoder_state = checkpoint.get('cond_encoder_state_dict', checkpoint.get('micro_encoder_state_dict'))
cond_encoder.load_state_dict(encoder_state)

denoiser.eval()
cond_encoder.eval()

print(f"Loaded denoiser, condition encoder, and diffusion model")

## Looped Generation for Conditioning Evaluation


In [ ]:
N_SAMPLES = 5000
GUIDANCE_SCALE = diffusion_config.GUIDANCE_SCALE
H, W = preprocess_config.TARGET_SHAPE

TRENDS = [-20, -10, 0, 10, 20]
REALIZED_VOLS = [100, 300, 500, 700, 900]
REGIMES = [
    {'interest_rate': 2.0, 'volatility_index': 15.0},
    {'interest_rate': 6.0, 'volatility_index': 40.0},
]

output_dir = REPO_ROOT / 'data/generated/ca'
output_dir.mkdir(parents=True, exist_ok=True)

shape = (N_SAMPLES, 1, H, W)
total = len(TRENDS) * len(REALIZED_VOLS) * len(REGIMES)
count = 0

for regime in REGIMES:
    for trend in TRENDS:
        for rv in REALIZED_VOLS:
            ir = regime['interest_rate']
            vix = regime['volatility_index']

            trend_t = torch.full((N_SAMPLES, 1), float(trend), dtype=torch.float32, device=DEVICE)
            realized_vol_t = torch.full((N_SAMPLES, 1), float(rv), dtype=torch.float32, device=DEVICE)
            interest_rate_t = torch.full((N_SAMPLES, 1), ir, dtype=torch.float32, device=DEVICE)
            volatility_idx_t = torch.full((N_SAMPLES, 1), vix, dtype=torch.float32, device=DEVICE)

            with torch.no_grad():
                cond_tokens = cond_encoder(
                    trend=trend_t,
                    realized_vol=realized_vol_t,
                    interest_rate=interest_rate_t,
                    volatility_index=volatility_idx_t,
                )

            samples_2d = diffusion.sample(
                model=denoiser,
                shape=shape,
                cond_emb=cond_tokens,
                guidance_scale=GUIDANCE_SCALE,
            )
            samples_1d = haar.inverse(samples_2d.squeeze(1))

            filename = f"t{int(trend)}r{int(rv)}i{int(ir)}v{int(vix)}.json"
            generated = {
                'conditions': {
                    'trend': trend,
                    'realized_vol': rv,
                    'interest_rate': ir,
                    'volatility_index': vix,
                    'guidance_scale': GUIDANCE_SCALE,
                },
                'samples': samples_1d.cpu().numpy().tolist(),
            }
            with open(output_dir / filename, 'w') as f:
                json.dump(generated, f)

            count += 1
            print(f"[{count}/{total}] Saved {filename}")

print(f"Done. {count} files saved to {output_dir}")